# Exploratory Data Analysis of the Selected PlantVillage Crop-Disease Subset

This notebook performs a reproducible EDA of the selected PlantVillage subset:
corn, tomato, potato, bell pepper, and soybean.

The analysis includes:
- Dataset integrity and cleaned evaluation partition creation
- Class and crop distributions
- Train–validation balance
- Healthy versus diseased composition
- Image metadata and colour statistics
- Visual sample inspection
- Exact and perceptual near-duplicate candidate audits
- Export of publication-ready figures, tables, and EDA findings

In [ ]:
import os
import re
import json
import random
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageOps
from tqdm.auto import tqdm

# -------------------------------
# Reproducibility
# -------------------------------
SEED = 7
random.seed(SEED)
np.random.seed(SEED)

# -------------------------------
# Global paper-style settings
# -------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,

    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.linewidth": 1.2,

    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,

    "legend.fontsize": 11,
    "legend.frameon": True,
    "legend.edgecolor": "0.4",

    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.85,
})

def paper_axes(ax):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.7)

    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

    ax.tick_params(which="both", direction="in", top=True, right=True)

# -------------------------------
# Paths
# -------------------------------
WORK_DIR = Path("/kaggle/working")
MANIFEST_PATH = WORK_DIR / "plantvillage_selected_manifest.csv"
OUTPUT_DIR = WORK_DIR / "eda_outputs"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

assert MANIFEST_PATH.exists(), (
    f"Manifest not found at: {MANIFEST_PATH}\n"
    "Run Notebook 00 manifest creation first."
)

print("Manifest:", MANIFEST_PATH)
print("EDA output directory:", OUTPUT_DIR)

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)

required_columns = {
    "image_path", "split", "crop", "disease",
    "label", "is_healthy", "sha256", "phash", "is_corrupt"
}

missing_columns = required_columns - set(manifest.columns)
assert not missing_columns, f"Missing manifest columns: {missing_columns}"

print("Original manifest shape:", manifest.shape)
display(manifest.head(3))

# Identify SHA-256 hashes occurring in the original training partition.
train_hashes = set(
    manifest.loc[manifest["split"].eq("train"), "sha256"]
    .dropna()
    .astype(str)
)

# Exclude only validation images with an identical image in training.
manifest["exclude_from_evaluation"] = (
    manifest["split"].eq("val")
    & manifest["sha256"].astype(str).isin(train_hashes)
).astype(int)

manifest["evaluation_split"] = manifest["split"]
manifest.loc[
    manifest["exclude_from_evaluation"].eq(1),
    "evaluation_split"
] = "excluded_exact_duplicate"

clean_df = manifest.loc[
    manifest["exclude_from_evaluation"].eq(0)
].copy()

clean_manifest_path = TABLE_DIR / "plantvillage_selected_manifest_clean.csv"
excluded_path = TABLE_DIR / "excluded_exact_train_val_duplicates.csv"

clean_df.to_csv(clean_manifest_path, index=False)
manifest.loc[manifest["exclude_from_evaluation"].eq(1)].to_csv(
    excluded_path, index=False
)

print(f"Original images: {len(manifest):,}")
print(f"Excluded validation duplicates: {manifest['exclude_from_evaluation'].sum():,}")
print(f"Clean images retained: {len(clean_df):,}")
print(f"Clean train images: {(clean_df['split'] == 'train').sum():,}")
print(f"Clean validation images: {(clean_df['split'] == 'val').sum():,}")
print(f"Clean classes: {clean_df['label'].nunique()}")
print(f"Corrupt images in clean dataset: {clean_df['is_corrupt'].sum()}")

assert clean_df["is_corrupt"].sum() == 0, "Unexpected corrupt images found."
assert clean_df["label"].nunique() == 20, "Expected 20 selected classes."